DEEP LEARNING Sistemas de recomendación basados en contenido

In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
import os

folder_path = '/content/drive/My Drive/Recomendadores'  # Ajusta si el nombre es diferente
os.listdir(folder_path)


['negocios.csv', 'test_reviews.csv', 'train_reviews.csv', 'usuarios.csv']

In [3]:
import pandas as pd
import numpy as np
import os
from datetime import datetime

# Función para cargar los datos (asumiendo que tienes los archivos CSV)
def cargar_datos():
    try:
        # Cargar los diferentes conjuntos de datos

        usuarios_df = pd.read_csv(os.path.join(folder_path, 'usuarios.csv'))
        negocios_df = pd.read_csv(os.path.join(folder_path, 'negocios.csv'))
        train_reviews_df = pd.read_csv(os.path.join(folder_path, 'train_reviews.csv'))
        test_reviews_df = pd.read_csv(os.path.join(folder_path, 'test_reviews.csv'))

        #usuarios_df = pd.read_csv('usuarios.csv')
        #negocios_df = pd.read_csv('negocios.csv')
        #train_reviews_df = pd.read_csv('train_reviews.csv')
        #test_reviews_df = pd.read_csv('test_reviews.csv')


        print(f"Datos cargados correctamente:")
        print(f"- Usuarios: {usuarios_df.shape[0]} registros")
        print(f"- Negocios: {negocios_df.shape[0]} registros")
        print(f"- Train reviews: {train_reviews_df.shape[0]} registros")
        print(f"- Test reviews: {test_reviews_df.shape[0]} registros")

        return usuarios_df, negocios_df, train_reviews_df, test_reviews_df

    except FileNotFoundError as e:
        print(f"Error al cargar los archivos: {e}")
        return None, None, None, None

# Función para extraer características de usuarios
def extraer_caracteristicas_usuarios(usuarios_df):
    # Convertir la fecha a características numéricas
    usuarios_df['yelping_since'] = pd.to_datetime(usuarios_df['yelping_since'])
    usuarios_df['años_en_yelp'] = (datetime.now() - usuarios_df['yelping_since']).dt.days / 365

    # Características de interacción social
    usuarios_df['num_amigos'] = usuarios_df['friends'].apply(lambda x: len(x.split(',')) if isinstance(x, str) and x.strip() else 0)

    # Características de actividad
    usuarios_df['ratio_useful'] = usuarios_df['useful'] / (usuarios_df['review_count'] + 1)
    usuarios_df['ratio_funny'] = usuarios_df['funny'] / (usuarios_df['review_count'] + 1)
    usuarios_df['ratio_cool'] = usuarios_df['cool'] / (usuarios_df['review_count'] + 1)

    # Características de popularidad
    usuarios_df['popularidad'] = usuarios_df['fans'] / (usuarios_df['review_count'] + 1)

    # Suma total de cumplidos recibidos
    usuarios_df['total_compliments'] = (
        usuarios_df['compliment_hot'] + usuarios_df['compliment_more'] +
        usuarios_df['compliment_profile'] + usuarios_df['compliment_cute'] +
        usuarios_df['compliment_list'] + usuarios_df['compliment_note'] +
        usuarios_df['compliment_plain'] + usuarios_df['compliment_cool'] +
        usuarios_df['compliment_funny'] + usuarios_df['compliment_writer'] +
        usuarios_df['compliment_photos']
    )

    return usuarios_df

# Función para extraer características de negocios
def extraer_caracteristicas_negocios(negocios_df):

    # Extraer número de categorías
    negocios_df['num_categorias'] = negocios_df['categories'].apply(
        lambda x: len(x.split(',')) if isinstance(x, str) and x.strip() else 0
    )

    return negocios_df


# Función para unificar los datasets
def unificar_datasets(usuarios_df, negocios_df, train_reviews_df, test_reviews_df):
    # Para el conjunto de entrenamiento
    print("Unificando conjunto de entrenamiento...")
    train_completo = train_reviews_df.copy()

    # Agregar características de usuario
    caracteristicas_usuario = [
        'review_count', 'useful', 'funny', 'cool', 'fans', 'average_stars',
        'años_en_yelp', 'num_amigos', 'total_compliments', 'ratio_useful',
        'ratio_funny', 'ratio_cool', 'popularidad'
    ]

    train_completo = pd.merge(
        train_completo,
        usuarios_df[['user_id'] + caracteristicas_usuario],
        on='user_id',
        how='left',
        suffixes=('', '_usuario')
    )

    # Agregar características del negocio
    caracteristicas_negocio = [
        'stars', 'review_count', 'is_open', 'num_categorias', 'address' , 'city',
        'state', 'postal_code', 'latitude', 'longitude' , 'attributes','is_open',
        'categories', 'hours'
    ]

    train_completo = pd.merge(
        train_completo,
        negocios_df[['business_id'] + caracteristicas_negocio],
        on='business_id',
        how='left',
        suffixes=('', '_negocio')
    )

    # Renombrar columnas para evitar confusión
    train_completo.rename(columns={
        'stars_negocio': 'promedio_estrellas_negocio',
        'stars': 'estrellas_review',
        'review_count_usuario': 'num_reviews_usuario',
        'review_count_negocio': 'num_reviews_negocio'
    }, inplace=True)

    # Para el conjunto de prueba
    print("Unificando conjunto de prueba...")
    test_completo = test_reviews_df.copy()

    # Conseguir user_id y business_id para cada review_id en test
    # Necesitamos esta información de train_reviews o alguna otra fuente
    id_mapping = train_reviews_df[['review_id', 'user_id', 'business_id']]
    test_completo = pd.merge(
        test_completo,
        id_mapping,
        on='review_id',
        how='left',
    )

    test_completo = test_completo.rename(columns={'user_id_x': 'user_id'})
    test_completo = test_completo.rename(columns={'business_id_x': 'business_id'})

    test_completo.drop(['business_id_y', 'user_id_y'], axis=1, inplace=True)


    print(test_completo.head())
    print(usuarios_df.head())
    # Agregar las mismas características que para el conjunto de entrenamiento
    test_completo = pd.merge(
        test_completo,
        usuarios_df[['user_id'] + caracteristicas_usuario],
        on='user_id',
        how='left',
        suffixes=('', '_usuario')
    )
    test_completo = pd.merge(
        test_completo,
        negocios_df[['business_id'] + caracteristicas_negocio],
        on='business_id',
        how='left',
        suffixes=('', '_negocio')
    )

    # Renombrar columnas igual que en entrenamiento
    test_completo.rename(columns={
        'stars_negocio': 'promedio_estrellas_negocio',
        'stars': 'estrellas_review',
        'review_count_usuario': 'num_reviews_usuario',
        'review_count_negocio': 'num_reviews_negocio'
    }, inplace=True)

    return train_completo, test_completo

In [4]:
usuarios_df, negocios_df, train_reviews_df, test_reviews_df = cargar_datos()

<ipython-input-3-ec3f98d1386a>:11: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  usuarios_df = pd.read_csv(os.path.join(folder_path, 'usuarios.csv'))


Datos cargados correctamente:
- Usuarios: 699619 registros
- Negocios: 30069 registros
- Train reviews: 967784 registros
- Test reviews: 414765 registros


In [5]:
# Preprocesar datos
print("Extrayendo características de usuarios...")
usuarios_df = extraer_caracteristicas_usuarios(usuarios_df)

Extrayendo características de usuarios...


In [7]:
print("Extrayendo características de negocios...")
negocios_df = extraer_caracteristicas_negocios(negocios_df)

Extrayendo características de negocios...


In [8]:
# Unificar datasets
train_completo, test_completo = unificar_datasets(
    usuarios_df, negocios_df, train_reviews_df, test_reviews_df
)

Unificando conjunto de entrenamiento...
Unificando conjunto de prueba...
                review_id                 user_id             business_id  \
0  ieYPmCImINjPzTDFmEKBKA  79F9QrQSet-b1yRCIM243Q  sXSUzImYOcRRI3xtG2M85g   
1  QIkJ8fZ4yx_QaHahWWszAA  chuM6TBkFHtTwJ6z96Hj1A  Ipt9ga67vVC_2ob3YmVwNA   
2  seR2KhblYMWg-k9zzN6aYA  hF68a0mpu97u0oaryFYhyg  _RG4IByyBR528CMc7DefJA   
3  BToo00Fi5pfJFA5MI2HM5g  G4yX5Q1tFfwSucFOmiyjdA  xxlbRiWWQkk-6LST3Hd12g   
4  FHJAzi1imodBit3RWK7zQA  Srqi1xb7exdB9uRHxDeEkw  LgGqdFLD7-ca0Z9F_q4Fuw   

   useful  funny  cool                                               text  \
0       1      0     1  Amazing coffee and chill atmosphere. The staff...   
1       4      0     2  I pass by this joint every time I make a run t...   
2       2      0     0  Came here when my kitten got very sick by the ...   
3       2      0     0  So I'll preface by saying we did have an overa...   
4       0      0     0  This place is a joke. Worst bar service ever. ...   

 

In [9]:
# Reducir al 20%
train_reducido = train_completo.sample(frac=0.1, random_state=42)
#test_reducido = test_completo.sample(frac=0.01, random_state=42)  #20% aleatorio reproducible



In [ ]:
#train_reducido.to_csv('train_df', index=False)
#test_completo.to_csv('test_df', index=False)

In [10]:
from sentence_transformers import SentenceTransformer
import pandas as pd

# Si usas una GPU y PyTorch
model_transformers = SentenceTransformer('paraphrase-MiniLM-L3-v2')
model_transformers = model_transformers.to('cuda')  # Mover a GPU si está disponible


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.83k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/314 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [11]:
# Función optimizada para convertir JSON a texto estructurado
def json_to_string(json_obj):
    import json

    if not json_obj or pd.isna(json_obj):
        return ""

    if isinstance(json_obj, str):
        try:
            json_obj = json.loads(json_obj)
        except:
            return str(json_obj)

    # Convertir el JSON a una cadena estructurada
    json_str = json.dumps(json_obj, sort_keys=True)
    return json_str

# Función optimizada para generar embeddings en lotes
def generate_embeddings_batch(texts_list):
    batch_size = 128
    all_embeddings = []

    for i in range(0, len(texts_list), batch_size):
        batch_texts = texts_list[i:i+batch_size]
        batch_embeddings = model_transformers.encode(batch_texts)
        all_embeddings.extend(batch_embeddings)

    return all_embeddings

# Aplicar a los campos JSON de forma optimizada
def process_json_fields(df):
    # Preparación para hours
    if 'hours' in df.columns:
        print("Procesando campo 'hours'...")
        df['hours_str'] = df['hours'].apply(json_to_string)
        hours_texts = df['hours_str'].tolist()

        # Generar embeddings en lote
        hours_embeddings = generate_embeddings_batch(hours_texts)
        df['hours_embedding'] = hours_embeddings
        df.drop('hours_str', axis=1, inplace=True)

    # Preparación para attributes
    if 'attributes' in df.columns:
        print("Procesando campo 'attributes'...")
        df['attributes_str'] = df['attributes'].apply(json_to_string)
        attr_texts = df['attributes_str'].tolist()

        # Generar embeddings en lote
        attr_embeddings = generate_embeddings_batch(attr_texts)
        df['attributes_embedding'] = attr_embeddings
        df.drop('attributes_str', axis=1, inplace=True)

    return df

# Función principal para aplicar todas las transformaciones
def aplicar_transformaciones(df):
    df_transformado = df.copy()

    # 1. Crear embeddings para la columna 'text'
    print("Generando embeddings para texto...")
    df_transformado['text'] = df_transformado['text'].fillna('')
    texts = df_transformado['text'].apply(str).tolist()

    # Procesar en lotes
    df_transformado['text_embedding'] = generate_embeddings_batch(texts)

    # 2. Procesar campos JSON
    df_transformado = process_json_fields(df_transformado)

    # 3. Continuar con el resto de transformaciones...

    return df_transformado

# Aplicar a train y test
train_transformado = aplicar_transformaciones(train_reducido)
test_transformado = aplicar_transformaciones(test_completo)

Generando embeddings para texto...
Procesando campo 'hours'...
Procesando campo 'attributes'...
Generando embeddings para texto...
Procesando campo 'hours'...
Procesando campo 'attributes'...


In [12]:
# train_df = train_reducido
train_df = train_transformado
test_df = test_transformado

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import StackingRegressor

# Extraer los embeddings como características
X_train_embeddings = np.concatenate(
    [np.stack(train_df[col]) for col in ['text_embedding', 'attributes_embedding', 'hours_embedding']],
    axis=1
)


# Seleccionar características numéricas adicionales
numerical_features = [
    'useful', 'funny', 'cool',              # Características de la reseña
    'review_count', 'useful_usuario', 'funny_usuario', 'cool_usuario',
    'fans', 'average_stars', 'años_en_yelp', 'num_amigos',
    'total_compliments', 'ratio_useful', 'ratio_funny', 'ratio_cool', 'popularidad',  # Usuario
    'num_reviews_negocio', 'is_open', 'num_categorias',  # Negocio
    'latitude', 'longitude'                 # Ubicación
]

# Verificar que las características numéricas existan en ambos conjuntos
valid_features = [feat for feat in numerical_features if feat in train_df.columns and feat in test_df.columns]
print(f"Características numéricas utilizadas: {valid_features}")

# Preparar características numéricas
X_train_numerical = train_df[valid_features].fillna(0).copy()

# Normalizar características numéricas
scaler = StandardScaler()
X_train_numerical_scaled = scaler.fit_transform(X_train_numerical)

# Combinar embeddings con características numéricas
X_train = np.hstack((X_train_embeddings, X_train_numerical_scaled))

# Variable objetivo
y_train = train_df['estrellas_review'].values

# Dividir conjunto de entrenamiento para validación
X_train_split, X_val, y_train_split, y_val = train_test_split(
    X_train, y_train, test_size=0.1, random_state=42
)


# Entrenar Random Forest
print("Entrenando modelo Random Forest...")
rf_model = RandomForestRegressor(
    n_estimators=50,       # número reducido para más velocidad
    max_depth=10,          # limitar profundidad para evitar sobreajuste/lentitud
    random_state=42,
    n_jobs=-1              # usa todos los núcleos
)
rf_model.fit(X_train_split, y_train_split)

# Validar
y_pred = ensemble_model.predict(X_val)
mae = mean_absolute_error(y_val, y_pred)
print(f"MAE en validación: {mae:.4f}")

Características numéricas utilizadas: ['useful', 'funny', 'cool', 'review_count', 'useful_usuario', 'funny_usuario', 'cool_usuario', 'fans', 'average_stars', 'años_en_yelp', 'num_amigos', 'total_compliments', 'ratio_useful', 'ratio_funny', 'ratio_cool', 'popularidad', 'num_reviews_negocio', 'is_open', 'num_categorias', 'latitude', 'longitude']
Entrenando ensemble...


In [ ]:
import torch
torch.cuda.empty_cache()
import gc
gc.collect()

9573

In [ ]:
test_df = generate_embeddings_batch(test_df, model_transformers)

Generando embeddings para texto...


In [ ]:
X_test_embeddings = np.concatenate(
    [np.stack(test_df[col]) for col in ['text_embedding', 'attributes_embedding', 'hours_embedding']],
    axis=1
)
X_test_numerical = test_df[valid_features].fillna(0).copy()
X_test_numerical_scaled = scaler.transform(X_test_numerical)
X_test = np.hstack((X_test_embeddings, X_test_numerical_scaled))

# Hacer predicciones en el conjunto de prueba
print("Realizando predicciones...")
predictions = rf_model.predict(X_test)

# Asegurarse de que las predicciones estén en el rango correcto (entre 1 y 5)
predictions = np.clip(predictions, 1, 5)

# Crear un DataFrame con los resultados
results_df = pd.DataFrame({
    'review_id': test_df['review_id'],
    'stars': predictions.flatten()
})

# Guardar resultados en CSV
output_file = 'predictions_ML.csv'
results_df.to_csv(output_file, index=False)
print(f"Predicciones guardadas en {output_file}")

# Mostrar las primeras predicciones
print("\nPrimeras 5 predicciones:")
print(results_df.head())

Realizando predicciones...


ValueError: Exception encountered when calling Sequential.call().

[1mInput 0 of layer "dense_6" is incompatible with the layer: expected axis -1 of input shape to have value 1174, but received input with shape (32, 406)[0m

Arguments received by Sequential.call():
  • inputs=tf.Tensor(shape=(32, 406), dtype=float32)
  • training=False
  • mask=None

In [ ]:
results_df['stars'] = results_df['stars'].round()
results_df.to_csv('./data/submissions/DL_rounded.csv', index=False)